# MatMul Bench — Web UI thực nghiệm nhân ma trận trên Google Colab Pro

Notebook **tất cả trong một** (đã nhúng sẵn `index.html`, `server.py`, `colab_tunnel.py`): chỉ cần upload file `.ipynb` này lên Google Colab Pro, chọn **Runtime → Change runtime type → GPU (T4 / L4 / A100)** rồi chạy lần lượt các cell bên dưới.

## Bước 1: Kiểm tra phần cứng GPU/CPU, clone repo & biên dịch CUDA

In [ ]:
import os, subprocess

# 1. Clone hoặc cập nhật repo
if not os.path.exists('/content/Matrix_Multiplication'):
    !git clone https://github.com/dthai2424/Matrix_Multiplication.git /content/Matrix_Multiplication
else:
    !git -C /content/Matrix_Multiplication pull

%cd /content/Matrix_Multiplication

# 2. Kiểm tra GPU & tự động nhận Compute Capability (T4=sm_75, A100=sm_80, L4=sm_89)
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!echo "CPU vCores: $(nproc)"

cap = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'], text=True
).strip().splitlines()[0].replace('.', '')
cuda_arch = f'sm_{cap}'
print(f'>>> Biên dịch với CUDA_ENABLED=1 CUDA_ARCH={cuda_arch}')
!make clean && make CUDA_ENABLED=1 CUDA_ARCH={cuda_arch}

## Bước 2: Giải nén bộ Web UI (`index.html`, `server.py`, `colab_tunnel.py`) & cài FastAPI

In [ ]:
import base64, gzip
from pathlib import Path

web_dir = Path('/content/Matrix_Multiplication/web_ui')
web_dir.mkdir(parents=True, exist_ok=True)

FILES_B64 = {
    "server.py": "H4sIAAU+CGsC/909y24kyXH3/orcGkOqHjabj+GORoRaFofkPLR8iexZrUDSNcWq7O5aVle16sGHOAQM+CAYvtjwwQdfvFgYhgEbkGADBmaOI+jofxh/iSPyUZVZldXd5MzKkmeXZFdWZGRkZERkROSjH3y2lKfJ0lkQLdHogkyus1EcPWpZltV65qbZxsFLcuZ65zTyySBOyK6bJcEV2c3DLJiEgedmQRyRpzTyRmM3OSc/j5PzM3wi9vM4HoaUbMahe0YOkpgskH4eRTQMomG723qV0pS45KD/CwIYdoIov1risNjOr2gSL57lgwFNAJwk1IUGx5SkGXwcY1E8IN2lsZuN89A5k+2vtxbJ9jjIUvJ6ErrRa2JnceaGUI1OUvI9skfCIM3aJRS+cNLMTbLXxM1INqKEXrleRsZBCKDUi6Hj1IX+nNMEaCcMNiX2ReDCZ399fRDm6UjBmECdxH9NgvGY+oGb0fCauCnH4YbDOAmy0ZgMgihIR8CBNCbeiKH0E/cSyLugJa7R9VkSAC4sJX7OWLG2iDSTzYNXS88PXgGdHnJKMgAhvgftTEbuQ5LlETyX6Pw4oq/JJRCA7QNbNo++JJzeFEbHjy+jMHZ99wyGrXsVpktdL71Y6o6ycUgGoTtMmVi0Bkk8Jo4zyLM8oY4DPZ3ESUbcKAJeozykrZYog/ry49dpHMnPcSo/wShlMN5j+fzLnOZUPiTFp5SG1MuKp/xsksQeTQs02QjEwsfOygIQFvk5zwOfEz1xs1EYnEmKD+CRv8iuJ8g5Ub4RXXfIVuBlHZDLFH7vT7BbbtgBCZ6EtNXKkuv1FoF/shvZdYteeXSSkZesZDtJ4mSdkAfk50EEjE3JwA1D1CRWDeBJj+zBeCAItE7XSTCMYugxJ2gAuudOAolfqGKHvOj3D7ZZQ0CQBtodB74f0ks3oV0Y0VRW3dw/PNotXulVEppOYLhoAfwsCOmhKMTGdnfKpyOperJI8PTad6Ms8CSKp25Kd2OfArOeBTT0W62nG0fbztbLQ+gxctwG0YFmHKeN7cfhBbXb3QnQFmWtw+2DfRU0TrtglYIkjrpDmtnW7kZ/99WOI8GsDrG6VltB1AoGBMSQ2AWmJWJVrYTV7tIrGNjUboPU+sSWFAoyZtThI6+QWqkOPX65t3H4C2dvY3cbXtdwASZqEaAUuhe5YNV6ABRlFqFhSg1Nt17ubW1/5eBwKK0hlSBb9IopKOilO5nAayEqdhZkIe1ZYLDBWHMLTaAcWHZBkxSkp2etdpe7y1YbK3Zd33dKCbJZF3XR6bAykOL40gErNgyitHdsPbRO1RdjCvOHb3gxAv2EduWLdqvV8kI3TclhHh1SUPs0swvRESxOg1/RdJ3p4HEQZafYOxQp26cDF2ag3vHnK6sdsrK8utYhq8trT047YA8jJ6TRMBv1VuDJvSqeVtsMKTcV09FCTcALSOfCmOQRoANMNUSPOmRIsRaMxOoyh/Zy33UyUABzlZXHHGw4yZ04Cq/XyVkchzWwZy6IimiepjRz+FTheIPh9BqS7cWU/dP4jHMbAMGqw7yUOQ4ffzYGNBx0iqev4zMn8NdxGi4LvbEvuAnFp0r5JZSjGpdFOCmvM9OKsB20tEoF7DNjD9gvWrB09fPHyxxGiIUkq8upARD+QX8JVMEb+F0pvmTFl5ViJAzK8Y/+QqcJQPQCHZhegPoD4Rabxro/w9/HcvI41rt9emoBNgXQbldI4lPcejH7HJfzXvcgntDoVMwhlR66kUfDkPrMGMCg66+F59HwFmxcBj3zqdKs0JF6S0LmuPwzAah0EWodn+p1gHcJGkkhsEoz9bpaiw+gBfCCwMSmNCFZAlMpTtkwkWVVBogWIilBOKrHFrMm1unx8imaXq2MG95lM5rCYHBkKxWoeAweQUZ9h/mYEqqCiztZTkp/2QAALif0juFwKHhrgFAdeNAr4wgwbxTadrN18NBiFzGj69PFXzZq+0/3nx6pGqfqPeK8uWUgzs7+5hdYWTpS3Z3YO2cYHpDFT/cPsL1wEx/nE/BT0d0ACwHTEYzkCxpOYIb4xO2hVQPXwYEAx02uHfQAYdZf/DEzTNyeYLwBSuMHPogSDI/mPChTeUederVyxIWWBTBrbxSDBSJXtFHxJOS/hIJDHZVgLaXQTBGMDvbPB/nzQKMEY8GGD2LeSV2nhMsKb6s2GAWhoMWKUwvEybqRznk3vQbBHNvtW1KWJeCSw3wNhVZpwS1vkkPdAkiYqzgBcoDL1qvoPIIoA6MXtVYYDyGUDLlJhfrgF3loZuM8ynjNFQV6MrpONXCYk22YZO16tVUYkCX4rdRO3LEzPINa4PsoxUNGuPXF6P1/gixORu+/ycgo+PDu1xHBMGvz1daGSvEFohkzNGrvwRjkGXU8d4LINMYUU0alzshNHd44M8fKGyGyqJDX8LomxYUYqb2jk9jxg8Ric3QhyALkttUSssi8a2uJZt5SnC6Ksaz5uMxfkqGO/IfqAjE805QmNEizk9GrzAZzE6M96Vl5Nlh8Au4nxdAo7Vk83gHgdBKCrwoYaxohiMV33NSlGLza1sHhdr/PVaBnGepIMT9GUUbh5hiwHdvqAQ0r7eOVU0CZBBPgI//7fev7bSOmM+jNefFGhHn7RzzE02pMwLOqshhVYAnEEumZj8HINxkA1erfhbMaVhBXdK0BMQSIKXUTb2Qn1l/wQoxCTtKH6/BjdxfafwbIsI0OwkLA1X+583JvW8cHPRQoDWPGeI+2AJkvwLrDJM4n9orkd5U86WBp5AEOwl5I6k78O5DH/cgG8ipmBCmFqdkWtQpq2x838mOYzT9m5Mv6HzHyYKyqjN2l4z6mxgqmnqQL50/nZSwgbGKrsK/ITuBg5NucqVBcshRmMRujNvKQBW9tCLfuymaNd3GObFPcY29EvXMHisEc2xqG4xrZVnQR+IG7mI4DxWAXbxcXwUdPrhfBRvdQTTowInFy3WWJxY5q8Y2VcRp0s56XXnSimAfB8CGHKCutVDjVH9MMQJOe0qmt7S/3Xu3sdGoi0+snOa0Ug/8H3e+tlcV1vcMMRJ5VrHuQpMjM44kEZwZ/gtYegDVjDa60NKodq31aFRMIk22Grk1+3COPmgRmKMwEAwWUDWByxpV6ytxdjh9sebvdZISUASpbWW1qRc7GCIpcrYE9INspMBc9RfQJuHHiLmQ8HseRyHeDz5ASG7z63A0Jc0gIzO4xJrJ9wvJmfCrHYQa21qkfOmF8qXClC480sdum+dHK1iwcH1Zn2myoOCGnIrKuQdMQMYb3w/iDtdW1BozuyvLyvXA+/uHKagPOi/vi/Hxl1dTzlN6He8JgFWnZEkdproQbj4h0px2XBJxzem2jbWFOG3Pc4a/IgUFL7B3zn2EaUOWAjX3uRiTLGRtSBnQGPhhqqFIEgSd/KmkTFLFXBTIvJmdupOMKvKaqoFoOBygQnF1jn6dW4BAl/RAD+qJFTMYyNlssiduMJFx1wkcljnMKzno8Uejm2QWlAB2JBcJxm/GKKgXSYRifuWETNBMGAVJUSSEEk51pqCJAWg185LLBkhzg+Kd5mKUOzB02/DCvnyfUmIiYEi7rkhRkn6xj8DxEyyI5I5aAZuVwtCmXrSIVLWA2yrYS9ER032QxDYZQGtFLlMgeCDAuhg2qgS9Oi5iWSy+62PwhK7AH7VrgEfhX4JyAZQQW0ygf0wQssc0RdPjiXG/FEBHUXC35L3Iu3FCbUgA7X2z4IvBG/VEee86exaLJ4o14Xm6bAwaRMTJjPYp38jgaOvz3LyuY+7wqL11pwM9sBl84ILZSNXezfgzq25a2wuRoF0Q6KfVwgtHJ64/i4HngRo6gAaJkc/0hVJykdQQvAprvudC158929g+OZmBJJ5T6+cRAB6DoA+uPOIBgSBOaC5oEg4AlNa2BG4SF/VBHk463gmgkuHOwcXS0vVWxp4a5lcvHj8kyM06cafBkFiZFmbruBHTCtxvh8N/N1LfMgwSLtM6kfjZoBJCM3DlghYhCDfFpjjowSwF8dd4qhLE9BwoEBhxFnXkoBU8WyUTOzwHOxRLTJezDHDWECGLGhH+ao46UN6gkP06vddv4ti55wp/40g1zaoiBiuWVOMqCSLioTUFT1dCzJyGi2mRTrh/Z8HP3uUbUmTnX0AhsEr3bXKNmnIqGPlGKieGVCSKDuRT9Y2CCDC0V9cCUf9KGpsCU0bEp73pbm+Sy+By7qiat3ljm1JjVY94G1DALyXmHXGBaPz7X019mcwcUHp9LNiBxF008sdgWE9Y261ddiPk4TzWDzeZP5JERNbfe+NzBOa1ZzUQKt6wznKNOaQdxqi7rynLk1bT6xWpuDUP5BnA8noqE81Ks4CgoeHkHZs8nU+sHY/BCcdXPgKN812HTZwOeun1qz4htjJZFjLqwLGd5EPoOvaJejnUdXHOzjSv7nYal+Y5hLZyXiUBdLdJWEdhwdFqNqyCWZe0GaCZImnhLYzeIut5k8v0UV/NxQ1jku4lf7sFw0hx4yvaRiXAeFdX1vBwdUDJJ4mGCORrcIMX4Nhg4pRuIm85s8VhEbhq50l9R66HDMeX5eHHllPyo0usyi1YCSh3UINtGyDROMlz0Y8ErTRwcJ9VmiaVgYbP43jq5lokMiVjowwa33AaQJw7LKD1BRWCLrxEmhHDDBl94VSgoa1VWSYUdlqLAmBFB9xGLbn2w5kKPrGCu5gh3loAtBjJlVNyE0YgEU1iM/jZ5SOw1rCMEj1P+SGFjWL6chQsoe8HmW8L6MsEFbdZ/lbhpuFYRBcs/PWchKIS27OmIRZdl/kEMIQ6eHTGLDgjK/DMbvwVeqKqxshbIV8jX+aAqC02l4VRGT19PY2xFT0x8NC56iU/q8p4gGtcRxUe1XaSZr7WXK3+ssFzj+uRL1ody++kRy9axJWuapMQeu5k3YtsQFCNCvscei82euCo5aX9iog63HRxxZ3P/cPuIJ/cx1QnzjZ1YfQgtSTRyIyYV6tLJSbrABGWTpbHaiGZ3o3/48itnr4pka2PvOem/eLVJXrzc3iO7G6R/uLFHvni5+QKL9zfJHmDtScwcWf/F4fbGVo0goBjgjmISYtSrUPOQEbJUhL04ds9wRy3EcszNQtM6nuCOWzdJcAPI0+1n0GWZUSCXI9xDIDbn4k6Uz9qto/7GYd85ONzfPegfidmEbdw8BqLAp80AusP2MBW/uO/JBMjWKD85Xln6wckpUjpyr0mfZ9vYLkeWY2t3eAoN/vTzD2//Bd5+ePfb8n2nWBWWfkgVPwamBPzXEWul78Zk00nogGHGv4j5w9tvYhB2cIs/vP33iJwHH9791Zh4ow/v/hVGmsMD3CShk4ZmFgD3PkwDuwcw9pjkQ6lgbZTZJ3iQIL//hpx9ePttNJt+BXEfU3lkZ/VkYeeRhp6n60r0EhDg7tTAU5Y7xBY04nlGsURfgM2HfPPgFTkRBvT5wasCt0gIwidhrBFwQWwFmAuxYqB32apRgVtNH8JjHfIOLXCjL+ohb8Eg6Q2JpKNsSFTgkJWGTlEDN+WmJsJzkCwUSVG/j7Z/5mzt721XFdyoHN2Hf/5VzGg5se3jE797utBOO6j84gE+8izRiTAfGzvP9w34+dzEu2xrUvzGIHpvqmLwpjLCb+rsftPIybY0n7ID2J03xk6IFyJlpVa6slrMtAlBAi8sOEv42YmCtS9+8dRhlqtmiF3QcT8mo4DmBKebEckCcCfwM5gENyDAXjSnRPRzhW1Z4XNyh7GGcHMruMxb2j5YMXL5aR57gOLE3uuJWusMcY/I3hDeX6KM6kmbnLxhzc4AQ0YoJKxWO8taX4U6Gxj8LGPjLAxaVhDrvXjU3ItHgOdLnAuLjuw6QCPvVwceNuUDUp/xd4WgQsGmWoAwL2U4VRQrHTIrRpzxHpw8VHlzYtdRqeOzubH5Ynurim3PndwJGY/DcEOhU550cYKBwxwX6ttfx2fr2p7ADsFYAeSe5VbKBSnce1hETlssB0gGhom6Oj+nMXn1EkR3OArhB+ZxvtNRztjKKZkidmLrzW7WAZjrDgndMzxLAH1OWQJIn9xVhxnqyA0Oah8qiZMyKqsnJYAb3Yb9vUo7QBZLLVeNuGpq28bVRWKvSAS45Z+tyrFybLiy31THoD+lwRAz5taNWi+6Xb8RdeETtAG/2Xt9h+qtVc3lIJBh/yn5rIcNrRu5ZIJn4EZovi+6W9uMMUciCM/G4F66UnytKZkQTIFrPJkr9TMzBS5S3yiPzTBMUAGKC2wzHJNkDG7w77TUkhw32Sd9HKf1DMMhUYvvc1ajptO7ZYD0fI/MGfMtKY6kyUFdNxkTLDcZkYpU3BTjHMbMI2G+8zqrfivyEbiFSYt6pLIjUJFVUVKRdW2WW7wMm7tk95SWZGA0TzuFwE1pollvtI3d0xSmriilgkCc3Kgg8yjG3QXuboJWCthUrosI8i5M19dbZ42uMSskD2T6KinS2Z2LFp1ZLAOlAZRHAOrv5pAKfbnW1MfKgqwEWa1I4D1Ei5NuEivxZr3BgvPVS733Zuszr+VWrPZKAwS31Twob91zSXG+ZURl6XBF3UzetFRY36bH9OKPTxUfYKIcFwbdMAQnbYnwUBg+sDBJ+G4mlSkCuFk6k7iXcgtDKczFW+Yh1ZaaZZ32DK1Yna0VjxSQ6h4ECbOmwJQmxjCrzOsTNvp6f1BL8v/KCEx13Wa5bd+FIZi6h+BP1BiIrAW/TkAkLBSdLxIX88yTlfN/2sEjufLr6KuyymTXaU1dfVUMQBUSMy4Oyhc7Lay/nNBk4PAVZ9SUytvMYYeKGEMNr1nlYXPl4bTKPLVhfhdklKeIkA/Hp6aazN9rqK2sCRte4xZkIMsb0frBo9uPdkWnxWpzuaKlABispQFezPkyY9tqDM4G1ubow9t/HpPf/R3PoW8evPqfv/x7TFzZQsz/+7cPO+SmFLrbTbKAma22Ca8M5iyZ0qPWH4cGNwZYZVbbdzPcdeAHXmYbVLN9O9VPL5KIZq1nC7QmhQ/4WncZCjZYhm4+wUOQM4Wu1OyptqKq6DWPwTQIiubX3Icm9EMD+rUG9MMG9J9X4f9PR3j1OxrhY2n+TqfGNX+wfj76zvqpmPJT444og1yzw1NzSfVYyNzUGVBAejXIR82yObccm1TqcyNsfauShH/8RyPzzeHLpxJ5Pm/PFvu6IBXsO50a9aSRi/FMIxO0VpRrHKRwYv32DL9tWlAxZWAQ9ewx4MsfsxxK3G52XXchNd+oxuEpjtKMSUH3muoOfLPLdjuL4awn95FzVvFWrvRcxgkE54i5noydNwOL7gbbTIB/Sw+kSMO64IMkzkATgTR0IerUy7QtvLg8k12resJ0Sb0AaRKnwZWlLzc8YPfDZUngnadkCBO9h7YrZjGIcj3cQrmOxBelKlfKfaafh5U96Kh0A3nsGAv8rWx3xWS3fryU3b5iXj7yxoad4N6l38MtWAzg0jcYxjTzg2jmYU8BiQc7JeVGAHZwtBHAC+MU36SGc6N6z/FCBQS2JbLKMpRh1FVeiL4gc+FTS3sPg4eHNfTFKL6ZB6mqrzolIbuKzYH/cTDYhXBd/sc+Lob0tIN6x36Wu6vG05IMkXm3dOMpIca2UR6dQ9PAFAxDbEWM1pZ/+Ni8pXraYebmOwYMG9BZ6/epL86Us/pdn+LdPnZto3xCQcU9fnMD+2RbJ8kJMwPwWyuVZearEmBUF3qsTeN7PsBYH7NlAN3cI1TxjhAT+C23rzOiGvavS2U1rkzxtajGcz5TaZm+gM6obJuPqCI53Ukchnbb7CTo1m4rcYEvbnQN8yJuJ0TjdnadGdahZ4rrNF3S9Gr1bkq1/Hl7Kj4hr8nq+sxjNM1C+1E6dzfNmZ8YIdwfpUkfZSTYceJm2lt1kZbHN+o4p+iJUq8+EXYv3SCrTJDMlZAXhwlr3+WOHZa0mg9Y/4GmVjZhKm28PNhunjdLuKP+1v6rfh2y4c4FISO4ENszLE5VJWTaxGuYPbUDoGlK+Y4br8u7pxqX2nkiXDeR56eUOncSComjm3DR4FL9CQWketKEXZp75enhlIpkcWW2u6zvJhhYx9uHh/uHp+QGMN9aIgxhd+GGurNaermNVttofQuPqUCgc2KWqmsqDnQULtY9yTA7bnNRoXFXuTVQuxYDT4DLG4R7hnPrQi3xrtLyLI0AwNuFFYuIJ/vE+aEClXIqUcEkSlmY14V31kevcOPNyKbEbSFvIhAqnk1JXnnvokzyymcT3tCdpJj+pR5A83tylHv7yCLDUN7u19HuMKusvGGwqYyDAa5kI4KWfG4GxdhU9EMpMcDjVb5p89ofDvE6mUMMDJeZaYiuwnQmIoSZiYhdltuEiV/FzS/UnYmJs3O6YE7BcXvndBcq/3dxIyNe8n4IEwL91Jcv/gRvFmbHEJcsvFOK3xvtsBtoe+rt0m2WvYBJDYxVHkh/RThw5e3HptPFbpBS/VpsG+8EzVOmp7215bUObiRwg7CnXJfMEA9YirWtnnxSibKVhptPG7dxRJSOupNgSd7EaPF+QVFxOaOkfXRZbnCoXNzYEn4NXjZbu/NPXDFtPqw15pbKwvviLe24laBnHdpVy1W7UD8H3iTUCgaz+ku1v5fKS1W/l5oX6n0v1ZYqfRd1vi2PnVWkQNCrCIEokTJwrzEubb7hlpip/P5UA190dRKnRV/zSO1nHtkJ3nVbXu4tuixuvTL0WtV4/mJOba/cmqaqfmVPljADuAb8+9+4JHv/b2OS4XLwtaTrhv+97ZIX77+9xgNWb7+5Jq/H7jm/7svZ3tt4urO91Vt5DY7X73/z4d0/el1F09rFHYDQ/W6xPaG4jWTlcYc8Wr2T+VouzVeJD09+/TWJRuw02MpjMoo/vP0vD1B3LUGCF1K8xAVPbWKAFaMjYd9E5SldJJC/Bmofr5Ef9fA8bQ+wPXqyJjxjjqTc+yTR8NPPmUQjAQDRCqLIGJ7VJ7faoKoUIRFFkTz2fT+mbLnRiKTvv/FGZI9cvP8nIs4SknN+lStw6tsJCT+8+3XJG6Z3QsgUZRPDVrtQXXNR0eLxG8SNOqJd+CvAG+77NXruRZ08gqjl3L6v486vPy9vO8QYmfdYtQOLe+o00bG6X8cB8jqxr/i9f1fszo1y5LTK2fyV5aEGtXqiVMdajPfgZWpAiyj0i/wehRp0oRBalVgly2wLOcCpOur1c+DAQrk8A3TA+0V2oFkI0awJXNwo33wHQkVLO5XC2mYXlciydHSpXFqoF+u3KouXgvri6nz8cpQu/lqDiWdEr47XV8TliACC3zahrCDZle8B6PE/2jcB9LQsDWZn4Ef/AoAe/mq69b+HlGs37eE1zuVXALDf7PKx4trycsSw6JgTdcq3araUiyXwVnNuHGwIrGAC6pXLZR0CJWmP5TraYFxcOo4jluNp8zDM1tzEG4u3wkMk5AHb5DN22ZK9RYQuAC/a5UoaW0Wruwn8K42WbjiiW2Ua5W9s5UsXhAVp6j8fMnzB8POKhRHGt0FaSSHcyX8Wt2OrEyeGKFIl2FkzDFecIY1wJRTv/VaujGvISBvtIN5gInbbiggIeySvNX1SvQhMGEb+dQrb40lmQHmN34dB2IZQOnHRRz6JTiLLtEakpjzW57gUupB348U/A96bGu+bcXFSB2yRF5xr/O6krp+PJym72QVv2Uvxu5fc1AsC8c0et7wzqpTWvrGnVODaMJX6yBZSHcyM9CyMeJYY7CIXRsW0ym90qbj+m7g8vrgJjEhi9MStKF7kS+YVX/urxQ3Po+HiU7mMy4GtmmvdNvibPLliUhr+5k9CaYrEtvKNHUV2TUihzALLnScyN12uKemk1ZRJrZTRBMQBd9Xd4bplzeypSS4k1WDP5NeILd0Mxpk6MvKFDeXqwMwRBo2hCVwL076mQASZTVM84dJbDX1EdNkQV0ItaEp+sd3SReR3x+ki8ijUAygRZjYEmLJxHm2qXwPAA8y63yhrwCQRRGWkJSQBOCZjCMGKewrj7/72w7u/AYfZh8AG+Fn6yP+Ad0u8+1aKJbqg7II+xRzgUQ3e+DHQI+5pYxNp4Q/Laqrfz0E+MnUjYzbv/X+wRsiNbKoSqxXjocRn2tStfsWZjZLGlo84kTBZS7Q9Ewd65Uc0SgF+YRLCOA7bweI4uGbrOJb23XD5RQCujNjixEr4ObXqd5sd7B/2UQyeLC8vy4UVURfdYhtY34FAL8161nKX/QfQiK+Hv9qt/wUpIyVXynEAAA==",
    "index.html": "...",
    "colab_tunnel.py": "H4sIAAU+CGsC/7VWzW4bNxC+71NMt5dVYlGy3fxUgAo4sZMYMWLBlhGgtrFY7XItIlxyQ3IdK0GAAnmB5tACPRRNG/TSokgPOVlHBX0PvUmHovZHcYKeSh3EJYfz930z5JdfdAqtOiMmOlScQz4xYyk2Pd/3vX1B27HMskgkwKNCxGOqIJUK7kt5xinclTwawUDJnrdO4NBEymi4F2mzNdiFQFN1ThXJJy2QAnKpDNzudrvE2yCwLZ8JLqNEQ8xlkaQ8UjQBlkLGtGbiDKxJmVOhIYJUUQp5MeIshgfD4eAQTCEE5cTbJGidCbRqxpXI2Jhc9zqda8SoSa2eYCRwdLAHkQYt0SP8ZwaYBkWjZEIWEXssWzgqdTlTtJzpcWEYr76KUa5kTHUlqSfV1LCsOlYoztmIKPq0oNp4qUI38siMcRGWIgP89Lw7W4c74fbuAfQXC0EYpozTMGzhWS35OQ1aJMdAhPEG+wdDFMPIA6kJwsaUFOSMmsC3W/4a+DbVfqvl3d3bP9q+t7d1sLMdDraGD0rt/gJ1LuOIL7Bv4OCvnrI560PgAQ6/TO4ZM+NiZHPaONlU0lGU00hT3eGRwcA7yRLyplCbM1FctKMsufmV77U8z0toCoh6oWjYkAta0P4GtFG9hRepLJAe/SUi5NmYxRjQagRWDvm0EHWn7FDUFEq41cUichbThrqu5AkPY26j2EIcfLy7BGLNijwO9x+2gHJNoYKwAyvulN4IaZYWCb1g2uigVfuWWyYH/vF6Z/MUPnwfYRWY+eVb1iwRQsgyNjtWqUXwE+NTjCJVPgJwbWm2Prt0Ix5nMgm68taNG25vmSHMdVAecahkERMOh0dSUOd2nIbIHczepyDzvDqq1D/esGE9HM+nvzD48Ho+/QnjK3uFaxVg1OwPAfF8+gPuvbBMflkH7GRCW3QW+qr8yMD2iaAK7BjrELNL48JEI06xFNqZLYjinMVSCTt1qnpRntuvdnsstbGz9Y1bpIu/dbdsaxNnNhXWl9bpWmUkfpb07XoJeKve0iaRhek3Hdwd7KzsU6Wa+4fD7f2jYS1h6IXpD1VB3ZIL37YUojmlebBBuhW/G1khueQcEcKOZnlWo2QHumSz1pB2fhLb/Oyh9BObjtS+f4Wie/Ppj7unFXxPxrP3iNiTVXA/vP7n7/n0bdw7sUlHdTX5HETMBOsrLPGPN2vuZ6gM7laEguGi50MQo5Wf4Xz2hsEZm/02Wdh7BWI8v/xLADaUJ+6SaNXUQZr+N20clxF4d7k4CgjZjgojizzBJuaWsMpwki46ITbCijM9R1j/9H+kgh2jItXsOe2vl+zwlq0gxCT1wV10Od4cgfKDsl0fR+3nW+1vu+2v26fXT1YvxhN7oLVMlbtDQ1SHuiyFHPnUpKZSpJEppkxqSZUG6ypJ+1RAQCheVB+J19rsyNCWC4BoGils5vZUa0XGvg0WrwJrpfZyVdGVCDJyphA9y7Mrco5ySE64Dn7fh2tw8/bnxFIf4PHOHTjaRXbOXsHh/PL9Izicfffofg/gRW3zpf9ZSwD32Xz6DmLKOYjZG8tcS3Qk9OWvE8yxxA9dzKevDfDZZQwJzSSBO/PL3zN8XMl8yXS8FN4Z8nk7y1BsVIvKS3mhxwsSuSP0Iqa5gYd0MpKRSnaFQRIWublyE50IV4nJfPqndc01aay88v1VFVjKRMR5gyMl2qg6wz2Db5ea9Y0+09zHx1cKYSiiDB890O+DH4b20glD3yl2N5D3LyZHYZCxCgAA",
}

for fname, b64_data in FILES_B64.items():
    content = gzip.decompress(base64.b64decode(b64_data))
    (web_dir / fname).write_bytes(content)
    print(f'[OK] Đã tạo {web_dir / fname} ({len(content):,} bytes)')

!pip install -q fastapi uvicorn
print('>>> Đã cài xong FastAPI & Uvicorn!')

## Bước 3: Khởi động Web UI & mở link HTTPS công khai (Cloudflare Tunnel)

Chờ vài giây, cell sẽ in ra đường link `https://xxxx.trycloudflare.com`. Mở link đó trên máy tính hoặc điện thoại để chạy thực nghiệm trực tiếp trên GPU Colab Pro (giữ cell này chạy trong suốt lúc demo).

In [ ]:
%cd /content/Matrix_Multiplication
!MATMUL_REPO_DIR=/content/Matrix_Multiplication python web_ui/colab_tunnel.py

## Bước 4 (Tùy chọn): Tải trực tiếp file kết quả về máy sau khi dừng tunnel

In [ ]:
from google.colab import files
import os

for fname in ['benchmark_results.xls', 'benchmark_results.csv', 'benchmark_charts.html', 'hybrid_alpha.cfg']:
    path = os.path.join('/content/Matrix_Multiplication', fname)
    if os.path.exists(path):
        print(f'Đang tải {fname}...')
        files.download(path)
    else:
        print(f'Chưa có {fname} (hãy chạy benchmark trên Web UI trước).')